# AIE S3 — Credit Risk

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-ai-engineering/challenges/aie-s3-credit-risk.ipynb)

**Classification.** 1,000 loan applications with 20 attributes each;
predict which applicants are **bad credit risks**.

Challenge: <https://ml-arena.com/viewchallenge/186>

---

**No code here, on purpose.** You have just worked through the
diabetes notebook, which ran the whole protocol on a continuous
target. This is the same protocol on a class label, and you write
it.

The dataset has been chosen so the Session 3 lesson repeats in an
even blunter form. You are going to fit a random forest that gets
**every single training applicant right** — and loses anyway.

---

## 0. Setup

Install the ML-Arena client. The distribution is **`mlarena-sdk`**
and it imports as `mlarena`; `pip install mlarena` is a different
package.

---

## 1. Get the data

Connect with your `mlk_user_...` key and download the dataset for
challenge **186** into the working directory.

---

## 2. Read it

Load the three files. Then answer, before modelling:

- What are **n** and **p**?
- How many columns are text rather than numeric?
- **What fraction of the training target is 1?** Compute it, and work
  out the accuracy you would get by predicting 0 for everybody. Write
  that number down — it is the floor every result must clear.

---

## 3. A quick look

Three or four plots, no more. This challenge is about validation, not
exploration.

- The class balance.
- The **bad-risk rate** per level of `checking_status` and of
  `credit_history` — the rate, not the raw count. `groupby` then
  `.mean()` on the target gives it directly.
- `duration` and `credit_amount` split by the target. Do longer, larger
  loans go bad more often?

---

## 4. Turn it into numbers

Thirteen of the twenty columns are categorical. One-hot encode them
and align the test columns to the training ones, or the two matrices
will not have the same shape.

Scale the numeric columns as well — `credit_amount` runs to five
figures while `installment_commitment` is single digits, and the
logistic solver will not converge nicely on that mix.

---

## 5. Walk into the trap first

Fit two models on **all** the training data and score them on **that
same data**:

- `LogisticRegression`
- `RandomForestClassifier(n_estimators=500)` — unrestricted

Report training accuracy for both. One of them will be **1.0000**.

Stop and say out loud what that number means before you continue. It
does not mean the model is perfect. It means the model has enough
capacity to store 700 answers, and you have just measured its
memory.

---

## 6. Now measure honestly

Two things, in this order.

**A held-out split.** Carve 25% off the training data, refit both
models on the rest, and report accuracy **and F1** on the part they
did not see. Put the training and validation numbers side by side and
look at the gap for each model.

**Then cross-validation.** Run 5-fold CV on the full training set for
both models, scoring `f1`. Report the mean and the standard
deviation.

Compare the two exercises. The single split gives you one number; CV
gives you five and their spread. If the gap between your two models
is smaller than the spread across folds, you have not actually
separated them — say so rather than picking the higher mean.

---

## 7. Fix the forest

The forest is not a bad model; it is an unconstrained one. Sweep
`min_samples_leaf` over something like `[1, 2, 5, 10, 20, 40]` and for
each value record **training F1** and **5-fold CV F1**.

Plot both against the parameter, on one pair of axes.

You should get two curves: one that falls the whole way, and one that
rises to a peak and then falls. Name which is which, and say which
value you would choose and why. That plot is the bias–variance
tradeoff drawn from your own data.

---

## 8. Choose the threshold

`predict()` cuts at probability 0.5. With 30% positives that is
usually the wrong place.

Take `predict_proba()` from your best model, sweep the threshold from
0.15 to 0.60, and plot F1 against it — **on your validation split or
your CV folds, never on the leaderboard**. Keep the best.

Then state the tradeoff in words: at your chosen threshold, what
happened to precision, and what did you buy with it? In a credit
setting, which of the two errors is more expensive?

---

## 9. Submit

Refit your chosen model on all 700 training rows, predict `X_test`,
apply your threshold, and write `submission.csv` with columns `id`
and `prediction`, where `prediction` is the integer 0 or 1. A
probability is rejected, not rounded for you.

Assert before uploading: one row per test id, ids unique, all values
in {0, 1}. Then submit and compare against the benchmark on the
challenge page (F1 = 0.5714).

Submit the unrestricted forest too, if you want to watch the
leaderboard agree with your cross-validation.

---

## 10. Write down what you found

Four sentences in the notebook:

- What training accuracy the unrestricted forest reached, and what it
  actually scored held out.
- Which model your cross-validation chose, and whether the leaderboard
  agreed.
- Which `min_samples_leaf` you picked and what the two curves looked
  like.
- What threshold you used, and which error you decided was worse.

If your CV ranking and your leaderboard ranking disagree, that is a
result worth reporting, not a mistake to hide — with 300 test rows,
differences smaller than your fold-to-fold spread are noise.